# 🧪 Agent Server 实时测试

测试**真实 agent server**（非离线），通过 HTTP 发送 RunAgentInput，验证工具调用链路。

| 模式 | 端口 | 说明 |
|------|------|------|
| 直连 | `:8000` | 跳过 APISIX，需手动传 `X-Org-Name` |
| APISIX | `:9080` | 走完整生产路径，APISIX 注入 `X-Org-Name` |

**前置**：`python3 -m uvicorn main:app --host 0.0.0.0 --port 8000 >> logs/agent.log 2>&1 &`

## 0. 初始化

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

from nb_utils import setup_env, get_token, make_payload, run_agent, print_events
setup_env()

import config
print('GATEWAY_URL:', config.GATEWAY_URL)

AGENT_DIRECT  = 'http://127.0.0.1:8000/copilotkit/admin'
AGENT_APISIX  = f'{config.GATEWAY_URL}/copilotkit/admin'
print('直连  :', AGENT_DIRECT)
print('APISIX:', AGENT_APISIX)

## 1. 登录获取 Token

In [ ]:
# ── 填入你的账号 ──────────────────────────────────────────
USERNAME = 'luke'
PASSWORD = 'jmx931228'
# ─────────────────────────────────────────────────────────

TOKEN, ORG = await get_token(USERNAME, PASSWORD)
print('ORG  :', ORG)
print('TOKEN:', TOKEN[:60], '...')

## 2. 测试：直连 agent :8000

In [ ]:
MSG = '列出我所有的项目'   # ← 修改这里

payload = make_payload(MSG, org=ORG, layer='org')
events  = await run_agent(AGENT_DIRECT, TOKEN, payload, org=ORG)  # 直连需要 X-Org-Name
print_events(events)

## 3. 测试：走 APISIX :9080（完整生产路径）

In [ ]:
MSG = '列出我所有的项目'   # ← 修改这里

payload = make_payload(MSG, org=ORG, layer='org')
events  = await run_agent(AGENT_APISIX, TOKEN, payload)  # APISIX 自动注入 X-Org-Name
print_events(events)

## 4. 测试：Project 层（需要 project_slug）

In [ ]:
PROJECT_SLUG = 'abcde'          # ← 填入要测试的项目 slug
MSG          = '列出所有数据库'   # ← 修改这里

payload = make_payload(MSG, org=ORG, layer='project', project_slug=PROJECT_SLUG)
events  = await run_agent(AGENT_APISIX, TOKEN, payload)
print_events(events)

## 5. 批量测试多条消息

In [ ]:
from nb_utils import make_payload, run_agent

CASES = [
    ('列出我所有的项目',                       'org',     ''),
    ('帮我跳转到 abcde 项目',                  'org',     ''),
    ('abcde 项目里有哪些数据库',               'project', 'abcde'),
]

for msg, layer, project in CASES:
    payload = make_payload(msg, org=ORG, layer=layer, project_slug=project)
    events  = await run_agent(AGENT_APISIX, TOKEN, payload)

    # 统计工具调用
    tools = [e.get('toolCallName','') for e in events if 'TOOL_CALL_START' in e.get('type','')]
    text  = ''.join(e.get('delta','') for e in events if 'TEXT_MESSAGE_CONTENT' in e.get('type',''))

    print(f'Q: {msg!r}')
    print(f'   tools : {tools or ["(none)"]}')
    print(f'   reply : {text[:100]}')
    print()

## 6. 查看最新日志（可选）

In [ ]:
import subprocess
LOG = os.path.join(os.path.abspath('..'), 'logs', 'agent.log')
result = subprocess.run(['tail', '-30', LOG], capture_output=True, text=True)

import json
for line in result.stdout.strip().split('\n'):
    try:
        d = json.loads(line)
        ev = d.get('event', '')
        if ev in ('tool.call.start', 'tool.call.end', 'tool.error', 'agent.output', 'agent.event'):
            ts = d.get('timestamp','')[-8:]  # HH:MM:SS
            if ev == 'tool.call.start':
                print(f'[{ts}] 🔧 {ev:20s}  {d.get("tool_name","")}  args={d.get("args_summary","")}')
            elif ev == 'tool.call.end':
                ok = '✅' if d.get('success') else '❌'
                print(f'[{ts}] {ok} {ev:20s}  {d.get("tool_name","")}  {d.get("duration_ms",0)}ms')
            elif ev == 'tool.error':
                print(f'[{ts}] ❌ {ev:20s}  {d.get("tool_name","")}  {d.get("error_type","")}:{d.get("error_msg","")[:80]}')
            elif ev == 'agent.output':
                print(f'[{ts}] 💬 {ev:20s}  {d.get("content_preview","")[:80]}')
            elif ev == 'agent.event':
                print(f'[{ts}] 📌 {ev:20s}  {d.get("event_type","")}')
    except:
        pass